# Modeling Population and Feature Design

This notebook converts the fairness-enriched Cook County property data into a leakage-aware housing-price modeling problem.

The design choices here are intentionally separated from model fitting. The goal is to establish **what observations belong in the prediction task, what constitutes a valid out-of-time test, and which variables may be used as predictors** before comparing algorithms.

### Objectives

1. Define an economically meaningful modeling population.
2. Inspect target quality and repeated-property structure.
3. Create a temporal holdout that distinguishes unseen from previously observed properties.
4. Exclude identifiers, post-sale information, assessment outputs, and fairness-audit variables from the predictor set.
5. Classify features by substantive type rather than raw pandas dtype.
6. Construct a training-only preprocessing pipeline for later models.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import build_analysis_dataset
from src.features import (
    TARGET_COL,
    FAIRNESS_ONLY_COLS,
    IDENTIFIER_COLS,
    POST_SALE_OR_FILTER_COLS,
    ASSESSMENT_COLS,
    LOW_QUALITY_COLS,
    ADDITIONAL_EXCLUSIONS,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    PRIMARY_FEATURES,
    define_modeling_population,
    create_temporal_splits,
    build_preprocessor,
    make_model_matrices,
)

DATA_ZIP = PROJECT_ROOT / "data" / "raw" / "cook_county_data.zip"

## 1. Reconstruct the Analysis Dataset

To keep this notebook reproducible on its own, the property–ACS merge is rebuilt through `src.data`. The Census API key is read from the environment when available; the credential itself is never stored in the notebook.

The resulting dataset includes the original property and transaction variables plus tract-level ACS attributes and fairness-group labels created in Notebook 1.


In [ ]:
data_fair = build_analysis_dataset(DATA_ZIP)

print(f"Observations: {len(data_fair):,}")
print(f"Columns: {data_fair.shape[1]}")
print(f"Years: {data_fair['Sale Year'].min()}–{data_fair['Sale Year'].max()}")
data_fair.head()

## 2. Target, Missingness, and Leakage Review

The prediction target is `Sale Price`. Before modeling, variables are separated into four categories that should **not** enter the predictive feature matrix:

- **Fairness-only attributes** — ACS socioeconomic/demographic variables and group labels reserved for post-model auditing.
- **Identifiers** — PIN, deed number, Census identifiers, and row IDs.
- **Post-sale/filter variables** — fields such as `Pure Market Filter` that are used to define the valid sample rather than predict its outcome.
- **Assessment outputs / low-quality fields** — variables that could create leakage, duplicate downstream valuation information, or have insufficient modeling quality.

This separation is important for the portfolio version of the project because it makes the modeling contract explicit: **demographic context is evaluated after prediction, not fed directly into the structural model.**


In [ ]:
missing_summary = (
    data_fair.isna().mean().mul(100).sort_values(ascending=False)
)

print("Columns with missing values (%):")
print(missing_summary[missing_summary > 0].head(20).round(3))

leakage_keywords = ["price", "sale", "market", "assessed", "assessment", "value"]
possible_leakage = [
    c for c in data_fair.columns
    if any(keyword in c.lower() for keyword in leakage_keywords)
]

print("\nTarget / leakage review candidates:")
print(possible_leakage)

## 3. Define the Modeling Population

The raw dataset contains nominal and non-market transactions that are inappropriate targets for a market-price model. In the original audit:

- all observations belong to the single-family (`SF`) modeling group;
- **167,384 of 204,792** observations are marked as pure-market transactions;
- **35,546** observations have a recorded sale price of `$1`;
- every `$1` transaction is classified as non-pure-market.

Rather than impose an arbitrary minimum-price threshold, the primary modeling population therefore uses the dataset's `Pure Market Filter == 1` definition.

After filtering, the minimum observed price is approximately `$10,003`. The remaining target is still strongly right-skewed, but the extreme upper tail appears to represent legitimate high-value properties rather than a repeated sentinel value. Those observations are retained, and later models are trained on `log(Sale Price)` while evaluation remains on the dollar scale.


In [ ]:
print("Modeling group counts:")
print(data_fair["Modeling Group"].value_counts(dropna=False))

print("\nPure-market filter:")
print(data_fair["Pure Market Filter"].value_counts(dropna=False))

print("\nNominal $1 transactions:", (data_fair[TARGET_COL] == 1).sum())

market_by_one_dollar = pd.crosstab(
    data_fair[TARGET_COL].eq(1),
    data_fair["Pure Market Filter"],
    margins=True,
)
market_by_one_dollar

In [ ]:
model_data = define_modeling_population(data_fair)

print(f"Pure-market observations: {len(model_data):,}")
print("\nSale-price distribution:")
print(model_data[TARGET_COL].describe())

print("\nSelected upper-tail frequencies:")
for cutoff in [1_000_000, 2_000_000, 5_000_000, 10_000_000]:
    n = model_data[TARGET_COL].gt(cutoff).sum()
    print(f"> ${cutoff:,}: {n:,} ({n / len(model_data) * 100:.3f}%)")

## 4. Temporal Holdout and Repeated-Property Leakage

A conventional random row split would be optimistic because the same property can appear in multiple sale records. The original diagnostics found:

- **167,191 unique PINs** among 204,792 observations;
- **32,432 properties** appear more than once;
- **70,033 observations** belong to repeated PINs.

The project therefore uses time as the first separation boundary:

- **Training:** pure-market sales from 2013–2018.
- **2019 primary test:** properties whose PIN was never observed in training.
- **2019 secondary test:** properties whose PIN appeared in 2013–2018.

The unseen-property set is the primary estimate of forward generalization. The repeated-property set is retained separately because it answers a different question: how well the model predicts a later transaction for a property it has indirectly encountered before.


In [ ]:
sales_per_pin = data_fair["PIN"].value_counts()

print(f"Unique PINs: {data_fair['PIN'].nunique():,}")
print(f"PINs appearing more than once: {(sales_per_pin > 1).sum():,}")
print(
    "Observations belonging to repeated PINs:",
    f"{data_fair['PIN'].isin(sales_per_pin[sales_per_pin > 1].index).sum():,}",
)

train_data, test_data, repeated_property_test = create_temporal_splits(model_data)

print(f"\nTraining observations: {len(train_data):,}")
print(f"Primary unseen-property 2019 test: {len(test_data):,}")
print(f"Repeated-property 2019 test: {len(repeated_property_test):,}")
print("Primary test PIN overlap with training:", test_data["PIN"].isin(set(train_data["PIN"])).sum())

## 5. Predictor Selection and Geographic Feature Policy

Feature selection follows substantive meaning rather than raw storage type.

### Excluded from all predictive models

- target and fairness-audit variables;
- row/property/deed/Census identifiers;
- post-sale or sample-selection fields;
- assessment estimates;
- free text and low-quality fields;
- redundant representations such as `Age Decade` and `Lot Size`.

### Geographic policy

The **primary structural model intentionally excludes fine-grained geography** (`Town Code`, `Neighborhood Code`, Census tract, latitude/longitude, etc.). Location is highly predictive of housing value, but it may also encode historical segregation and socioeconomic differences.

A later location-aware model will add geography as a benchmark. This design makes the tradeoff measurable rather than implicit: we can compare the gain in predictive performance with any change in subgroup error disparities.


In [ ]:
all_exclusions = list(dict.fromkeys(
    [TARGET_COL]
    + FAIRNESS_ONLY_COLS
    + IDENTIFIER_COLS
    + POST_SALE_OR_FILTER_COLS
    + ASSESSMENT_COLS
    + LOW_QUALITY_COLS
    + ADDITIONAL_EXCLUSIONS
))

print(f"Numeric structural features ({len(NUMERIC_FEATURES)}):")
print(NUMERIC_FEATURES)

print(f"\nCategorical structural features ({len(CATEGORICAL_FEATURES)}):")
print(CATEGORICAL_FEATURES)

print(f"\nTotal primary structural features: {len(PRIMARY_FEATURES)}")
print("Fairness attributes used as predictors:", sorted(set(PRIMARY_FEATURES) & set(FAIRNESS_ONLY_COLS)))

## 6. Modeling Matrices and Training-Only Preprocessing

The target is modeled on the natural-log scale to reduce the influence of the strongly right-skewed sale-price distribution. Predictions will be transformed back to dollars before computing accuracy and fairness metrics.

Preprocessing is defined as part of the later model pipeline so it is **fit only on historical training data**:

- numeric variables: median imputation;
- categorical variables: most-frequent imputation plus one-hot encoding;
- unseen categories at prediction time: ignored rather than causing failure.

This avoids preprocessing leakage from the 2019 holdout.


In [ ]:
matrices = make_model_matrices(train_data, test_data, repeated_property_test)
preprocessor = build_preprocessor()

for name in ["X_train", "X_test", "X_test_repeated"]:
    print(f"{name}: {matrices[name].shape}")

print("\nTraining price range:")
print(f"${matrices['y_train'].min():,.0f} – ${matrices['y_train'].max():,.0f}")
print("Non-positive training prices:", matrices["y_train"].le(0).sum())
print("\nPreprocessor:")
preprocessor

## 7. Modeling Design Summary

The final modeling design deliberately separates **prediction**, **generalization testing**, and **fairness auditing**:

| Component | Decision |
|---|---|
| Modeling population | Pure-market single-family sales |
| Training period | 2013–2018 |
| Primary test | 2019 properties unseen by PIN during training |
| Secondary test | 2019 repeat-sale properties |
| Modeling target | `log(Sale Price)` |
| Evaluation scale | Original dollars |
| Primary predictors | Structural property + transaction-time features |
| Geography | Excluded from primary model; added later as benchmark |
| ACS demographic variables | Audit only; never direct structural-model predictors |
| Preprocessing | Fit on training data only |

These decisions establish a leakage-aware baseline for the model comparisons in the next notebook. The next stage begins with a naive benchmark and then evaluates structural and location-aware Random Forest models using the same test design.
